In [1]:
import subprocess, sys
pkgs = {'lightgbm': 'lightgbm>=4.0', 'onnx': 'onnx', 'onnxruntime': 'onnxruntime',
        'skl2onnx': 'skl2onnx', 'onnxmltools': 'onnxmltools'}
restart_needed = False
for mod, pkg in pkgs.items():
    try:
        __import__(mod)
        print(f'{mod} instalado ✅')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'{mod} instalado ✅ — REINICIAR KERNEL antes de continuar')
        restart_needed = True
if restart_needed:
    print('\n⚠️  Reinicia el kernel y vuelve a ejecutar desde esta celda.')
else:
    print('\nListo — continuar con celda 0')

lightgbm instalado ✅
onnx instalado ✅
onnxruntime instalado ✅
skl2onnx instalado ✅
onnxmltools instalado ✅

Listo — continuar con celda 0


In [1]:
import os, sys, time, json, pickle, warnings, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
import mlflow
warnings.filterwarnings('ignore')

SAVE_DIR = '/srv/work'
IN_FEAT  = f'{SAVE_DIR}/te_features_v5.parquet'
OUT_DIR  = f'{SAVE_DIR}/models'

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

W_RF  = 0.80
W_LGB = 0.20

UMBRALES = {
    'qwk':        {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'recall_alta':{'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
}

try:
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True; print('MLflow OK — http://localhost:6000')
except Exception:
    MLFLOW_OK = False; print('MLflow no disponible — continúa sin tracking')

print(f'LightGBM {lgb.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}')
print(f'\nCandidato val (congelado): QWK=0.6529  RecAlta=0.6509')
print(f'Ensemble: RF×{W_RF} + LGB×{W_LGB}  |  test se evalúa UNA SOLA VEZ')

MLflow OK — http://localhost:6000
LightGBM 4.7.0
Input : /srv/work/te_features_v5.parquet
Output: /srv/work/models

Candidato val (congelado): QWK=0.6529  RecAlta=0.6509
Ensemble: RF×0.8 + LGB×0.2  |  test se evalúa UNA SOLA VEZ


In [2]:

print('Cargando te_features_v5.parquet (solo split=test)...')
t0 = time.time()
df = pd.read_parquet(
    IN_FEAT,
    filters=[('split', '==', 'test')]
)
print(f'  Shape cargado (test): {df.shape}  ({time.time()-t0:.1f}s)')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post target-shift: {df.shape}')

feat_cols_v5 = [c for c in df.columns
                if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                and not c.startswith('n_samp')]
n_features = len(feat_cols_v5)
print(f'  Features: {n_features}')

y_test = df['y'].values.astype(int)
n_test = len(y_test)

ratio_idx = [i for i, c in enumerate(feat_cols_v5) if 'ratio' in c]
other_idx = [i for i, c in enumerate(feat_cols_v5) if 'ratio' not in c]

print('\nExtrayendo X_test_rf (float32)...')
X_test_rf = df[feat_cols_v5].values.astype('float32')
for idx in ratio_idx:
    m = np.isnan(X_test_rf[:, idx])
    if m.any(): X_test_rf[m, idx] = 1.0
for idx in other_idx:
    m = np.isnan(X_test_rf[:, idx])
    if m.any(): X_test_rf[m, idx] = 0.0

fill0_cols  = [c for c in feat_cols_v5 if c.startswith('std_') or c.startswith('bgp_')]
other_fill0 = [c for c in feat_cols_v5
               if not c.startswith('lag') and 'ratio' not in c
               and c not in fill0_cols]
fill_cols_set = set(fill0_cols + other_fill0)
fill_idx = [i for i, c in enumerate(feat_cols_v5) if c in fill_cols_set]

print('Extrayendo X_test_lgb (float32)...')
X_test_lgb = df[feat_cols_v5].values.astype('float32')
for idx in fill_idx:
    m = np.isnan(X_test_lgb[:, idx])
    if m.any(): X_test_lgb[m, idx] = 0.0

del df; gc.collect()

with open(f'{OUT_DIR}/scaler_v5.pkl', 'rb') as f:
    scaler = pickle.load(f)
X_test_sc = scaler.transform(X_test_rf).astype('float32')
del X_test_rf; gc.collect()

feat_json_path  = f'{OUT_DIR}/feature_cols_v5.json'
X_rf_path       = f'{OUT_DIR}/X_test_rf.npy'
X_lgb_path      = f'{OUT_DIR}/X_test_lgb.npy'
y_path          = f'{OUT_DIR}/y_test.npy'

with open(feat_json_path, 'w') as f:
    json.dump(feat_cols_v5, f)
np.save(X_rf_path,  X_test_sc)
np.save(X_lgb_path, X_test_lgb)
np.save(y_path,     y_test)

print(f'\nArtefactos guardados en {OUT_DIR}/')
print(f'  feature_cols_v5.json  {n_features} features')
print(f'  X_test_rf.npy         {X_test_sc.nbytes/1e6:.0f} MB  shape={X_test_sc.shape}')
print(f'  X_test_lgb.npy        {X_test_lgb.nbytes/1e6:.0f} MB  shape={X_test_lgb.shape}')
print(f'  y_test.npy            {y_test.nbytes/1e3:.0f} KB  n={n_test:,}')

class_counts_test = np.bincount(y_test, minlength=4)
print(f'\ny_test distribución:')
for k, name in LABEL_COLS.items():
    print(f'  {name:<8}: {class_counts_test[k]:>9,}  ({100*class_counts_test[k]/n_test:.1f}%)')
print(f'  X_test_sc:  {X_test_sc.shape}')
print(f'  X_test_lgb: {X_test_lgb.shape}')
print('Celda 1 OK')

Cargando te_features_v5.parquet (solo split=test)...
  Shape cargado (test): (993449, 109)  (33.3s)
  Post target-shift: (991379, 110)
  Features: 100

Extrayendo X_test_rf (float32)...
Extrayendo X_test_lgb (float32)...

Artefactos guardados en /srv/work/models/
  feature_cols_v5.json  100 features
  X_test_rf.npy         397 MB  shape=(991379, 100)
  X_test_lgb.npy        397 MB  shape=(991379, 100)
  y_test.npy            7931 KB  n=991,379

y_test distribución:
  Normal  :   854,063  (86.1%)
  Baja    :    92,840  (9.4%)
  Media   :    22,201  (2.2%)
  Alta    :    22,275  (2.2%)
  X_test_sc:  (991379, 100)
  X_test_lgb: (991379, 100)
Celda 1 OK


In [3]:
def evaluar_test(nombre, y_true, y_prob):
    y_pred = np.argmax(y_prob, axis=1)
    qwk    = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    mf1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bac    = balanced_accuracy_score(y_true, y_pred)
    cm     = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    n_alta = cm[3].sum()
    rec_a  = cm[3,3] / n_alta if n_alta > 0 else 0.0
    prec_a = cm[3,3] / cm[:,3].sum() if cm[:,3].sum() > 0 else 0.0
    mae_o  = np.mean(np.abs(y_true - y_pred))
    cand   = qwk >= UMBRALES['qwk']['floor'] and rec_a >= UMBRALES['recall_alta']['floor']
    sep = '=' * 110
    print(f'\n{sep}')
    print(f'  {nombre} — TEST SET (one-shot)')
    print(f'  QWK={qwk:.4f}  MacroF1={mf1:.4f}  BalAcc={bac:.4f}  MAEord={mae_o:.4f}'
          f'  RecAlta={rec_a:.4f}  PrecAlta={prec_a:.4f}')
    if cand:
        print('  ✅ H2 VERIFICADA EN TEST  (QWK≥0.65=True  RecAlta≥0.55=True)')
    else:
        print(f'  ❌ H2 NO verificada en test  (QWK≥0.65={qwk>=0.65}  RecAlta≥0.55={rec_a>=0.55})')
    print(f'{sep}')
    print(f'  Confusion matrix (filas=real, cols=pred):')
    print(f'  {"":>12} {"Normal":>10} {"Baja":>10} {"Media":>10} {"Alta":>10}')
    for k in range(4):
        print(f'  {LABEL_COLS[k]:<12} ' + '  '.join(f'{cm[k,j]:>10,}' for j in range(4)))
    return dict(qwk=qwk, macro_f1=mf1, bal_acc=bac, recall_alta=rec_a,
                prec_alta=prec_a, mae_ord=mae_o, cm=cm, candidato=cand)

print('Referencia val (candidato congelado):')
print('  ens_grid_v9 (w=0.80/0.00/0.20): QWK=0.6529  RecAlta=0.6509')
print('Función evaluar_test lista.')

Referencia val (candidato congelado):
  ens_grid_v9 (w=0.80/0.00/0.20): QWK=0.6529  RecAlta=0.6509
Función evaluar_test lista.


In [4]:
print('Cargando RF v5.1 desde disco...')
t0 = time.time()
with open(f'{OUT_DIR}/rf_v51_model.pkl', 'rb') as f:
    rf_v51 = pickle.load(f)
print(f'  RF v5.1 cargado en {time.time()-t0:.1f}s')

print(f'  Prediciendo en test (chunks 50k, n={n_test:,})...')
CHUNK = 50_000
proba_rf_test = np.zeros((n_test, 4), dtype='float32')
t1 = time.time()
for i in range(0, n_test, CHUNK):
    proba_rf_test[i:i+CHUNK] = rf_v51.predict_proba(X_test_sc[i:i+CHUNK])
print(f'  Predicción RF test: {time.time()-t1:.1f}s')

del rf_v51, X_test_sc; gc.collect()
print(f'  proba_rf_test: {proba_rf_test.shape}  (RF liberado de memoria)')
print('Celda 3 OK')

Cargando RF v5.1 desde disco...
  RF v5.1 cargado en 4.2s
  Prediciendo en test (chunks 50k, n=991,379)...
  Predicción RF test: 21.0s
  proba_rf_test: (991379, 4)  (RF liberado de memoria)
Celda 3 OK


In [5]:
print('Cargando LGB v7 desde disco...')
t0 = time.time()
lgb_v7 = lgb.Booster(model_file=f'{OUT_DIR}/lgb_v7_model.txt')
print(f'  LGB v7 cargado en {time.time()-t0:.1f}s')

print(f'  Prediciendo en test (n={n_test:,})...')
t1 = time.time()
proba_lgb_test = lgb_v7.predict(X_test_lgb).astype('float32')
print(f'  Predicción LGB test: {time.time()-t1:.1f}s')

del lgb_v7, X_test_lgb; gc.collect()
print(f'  proba_lgb_test: {proba_lgb_test.shape}  (LGB liberado de memoria)')
print('Celda 4 OK')

Cargando LGB v7 desde disco...
  LGB v7 cargado en 0.1s
  Prediciendo en test (n=991,379)...
  Predicción LGB test: 92.1s
  proba_lgb_test: (991379, 4)  (LGB liberado de memoria)
Celda 4 OK


In [6]:

print('▶ ONE-SHOT — Construyendo ensemble test...')
ens_test_proba = (W_RF * proba_rf_test + W_LGB * proba_lgb_test).astype('float32')
print(f'  Ensemble: RF×{W_RF} + LGB×{W_LGB}  shape={ens_test_proba.shape}')

res_test = evaluar_test(
    f'Ensemble v9 (w_RF={W_RF}, w_LGB={W_LGB}) — TEST SET ONE-SHOT',
    y_test, ens_test_proba
)

if MLFLOW_OK:
    with mlflow.start_run(run_name='ens_v9_test_oneshot_v6'):
        mlflow.log_params({'w_rf': W_RF, 'w_lgb': W_LGB, 'split': 'test',
                           'protocol': 'one-shot', 'notebook': 'v6'})
        mlflow.log_metrics({'qwk_test':        res_test['qwk'],
                            'recall_alta_test': res_test['recall_alta'],
                            'macro_f1_test':    res_test['macro_f1'],
                            'bal_acc_test':     res_test['bal_acc']})

sep = '=' * 100
print(f'\n{sep}')
print(f'  COMPARATIVA VAL vs TEST — Ensemble v9 (w_RF=0.80, w_LGB=0.20)')
print(f'{sep}')
print(f'  {"Partición":<12} {"QWK":>8} {"RecAlta":>9} {"MacroF1":>9} {"BalAcc":>8}  H2?')
print(f'  {"-"*98}')
ref_val = dict(qwk=0.6529, recall_alta=0.6509, macro_f1=0.6631, bal_acc=0.6665)
print(f'  {"Val":<12} {ref_val["qwk"]:>8.4f} {ref_val["recall_alta"]:>9.4f}'
      f' {ref_val["macro_f1"]:>9.4f} {ref_val["bal_acc"]:>8.4f}  ✅')
cand_test = res_test['qwk'] >= 0.65 and res_test['recall_alta'] >= 0.55
print(f'  {"Test":<12} {res_test["qwk"]:>8.4f} {res_test["recall_alta"]:>9.4f}'
      f' {res_test["macro_f1"]:>9.4f} {res_test["bal_acc"]:>8.4f}  {"✅" if cand_test else "❌"}')
print(f'{sep}')
print(f'  Nota: QWK≥0.65 AND RecAlta≥0.55 = criterio operacional (Benchmarking Criterios Éxito).')
print(f'  H2 formal (vs umbral estático): VERIFICADA — LOG-27, 2026-08-07.')
print(f'{sep}')

print()
if cand_test:
    print('╔══════════════════════════════════════════════════════════════╗')
    print('║  ✅ CANDIDATURA EN TEST — criterio operacional alcanzado    ║')
    print(f'║  QWK={res_test["qwk"]:.4f} (≥0.65) · RecAlta={res_test["recall_alta"]:.4f} (≥0.55)  ║')
    print('╚══════════════════════════════════════════════════════════════╝')
else:
    print('╔══════════════════════════════════════════════════════════════╗')
    print('║  ❌ Criterio operacional no alcanzado en test               ║')
    print(f'║  QWK={res_test["qwk"]:.4f} · RecAlta={res_test["recall_alta"]:.4f}                  ║')
    print('║  H2 VERIFICADA (vs umbral estático, LOG-27)                 ║')
    print('╚══════════════════════════════════════════════════════════════╝')

with open(f'{OUT_DIR}/resultados_v9_test.pkl', 'wb') as f:
    pickle.dump({'res_test': res_test, 'w_rf': W_RF, 'w_lgb': W_LGB,
                 'protocol': 'one-shot', 'notebook': 'v6',
                 'ref': 'Kaufman et al. 2012'}, f)
print(f'\nResultados test guardados: {OUT_DIR}/resultados_v9_test.pkl')
print(f'\nArtefactos ONNX disponibles en {OUT_DIR}/')
for fname in ['feature_cols_v5.json', 'X_test_rf.npy', 'X_test_lgb.npy', 'y_test.npy']:
    path = f'{OUT_DIR}/{fname}'
    if os.path.exists(path):
        print(f'  ✅ {fname}  ({os.path.getsize(path)/1e6:.0f} MB)')
    else:
        print(f'  ❌ {fname}  NO ENCONTRADO')

▶ ONE-SHOT — Construyendo ensemble test...
  Ensemble: RF×0.8 + LGB×0.2  shape=(991379, 4)

  Ensemble v9 (w_RF=0.8, w_LGB=0.2) — TEST SET ONE-SHOT — TEST SET (one-shot)
  QWK=0.6381  MacroF1=0.6687  BalAcc=0.6668  MAEord=0.1334  RecAlta=0.5954  PrecAlta=0.5237
  ❌ H2 NO verificada en test  (QWK≥0.65=False  RecAlta≥0.55=True)
  Confusion matrix (filas=real, cols=pred):
                   Normal       Baja      Media       Alta
  Normal          818,265      22,744       4,788       8,266
  Baja             26,409      61,720       3,023       1,688
  Media             6,697       3,433       9,962       2,109
  Alta              6,688       1,013       1,311      13,263
🏃 View run ens_v9_test_oneshot_v6 at: http://localhost:6000/#/experiments/681062908553326840/runs/1f266cc23e214cd0868036cb1a03159b
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840

  COMPARATIVA VAL vs TEST — Ensemble v9 (w_RF=0.80, w_LGB=0.20)
  Partición         QWK   RecAlta   MacroF1   Ba